In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


Q5_FILES = [
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2017~2018/2017_data_179_activities.csv",
        "year": 3,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2018~2019/2018_data_179_activities.csv",
        "year": 4,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/1920_london32_stable179.csv",
        "year": 5,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/2021_london32_stable179.csv",
        "year": 6,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year7_179activities.csv",
        "year": 7,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year8_179activities.csv",
        "year": 8,
    },
]

OUTPUT_DIR = Path("/Users/zsh/Downloads/PROJECT/code")

In [2]:
Q5_COLUMNS = [
    "year",
    "LA_2023",
    "Age16plus",
    "Age9",
    "VolFrqB_Pop",
    "MEMS7GR_ALL",
    "Number_Activities_150",
    "wt_final",
    "wt_final_online",
    "wt_online_time",
    "wt_final_AB",
    "wt_final_AC",
    "wt_final_B",
    "wt_final_C",
    "wt_time",
    "mode",
]

MISSING_CODES = [-99, -98, -97, -96, -95, -94, -93, -92, -91]

WEIGHT_COL = "wt_final_online"

合并6年respondent level data

In [3]:
def load_q5_master(file_info_list, output_path=None):
    dfs = []

    for file_info in file_info_list:
        file_path = Path(file_info["path"])
        year_value = file_info["year"]

        df = pd.read_csv(file_path)

        keep_cols = [
            col for col in Q5_COLUMNS
            if col in df.columns
        ]

        missing_cols = [
            col for col in Q5_COLUMNS
            if col not in df.columns
        ]

        if missing_cols:
            print(f"Missing columns in {file_path.name}: {missing_cols}")

        df = df[keep_cols].copy()

        # 统一覆盖 year：2017=3, 2018=4, 19_20=5, 20_21=6, year7=7, year8=8
        df["year"] = year_value

        dfs.append(df)

    master_df = pd.concat(dfs, ignore_index=True)

    # 把缺失码转成 NaN
    master_df = master_df.replace(MISSING_CODES, np.nan)

    # 只保留 16+ 样本
    master_df = master_df[master_df["Age16plus"] == 1].copy()

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        master_df.to_csv(output_path, index=False)

    print(
        f"Done: {len(master_df)} rows, "
        f"{len(master_df.columns)} columns, "
        f"{master_df['year'].nunique()} years"
    )

    return master_df

q5_master = load_q5_master(
    file_info_list=Q5_FILES,
    output_path=OUTPUT_DIR / "q5_respondent_master.csv"
)

print(q5_master.shape)

print("Years:")
print(q5_master["year"].value_counts().sort_index())

print("Number of boroughs:")
print(q5_master["LA_2023"].nunique())

print("Age9:")
print(q5_master["Age9"].value_counts(dropna=False).sort_index())

print("MEMS7GR_ALL:")
print(q5_master["MEMS7GR_ALL"].value_counts(dropna=False).sort_index())

Done: 96629 rows, 16 columns, 6 years
(96629, 16)
Years:
year
3    15967
4    15889
5    16091
6    16028
7    16139
8    16515
Name: count, dtype: int64
Number of boroughs:
32
Age9:
Age9
2.0     7883
3.0    18022
4.0    19933
5.0    16445
6.0    14753
7.0    11772
8.0     5556
9.0     1386
NaN      879
Name: count, dtype: int64
MEMS7GR_ALL:
MEMS7GR_ALL
0.0    21824
1.0    10351
2.0    64454
Name: count, dtype: int64


构建q5需要的分析变量

In [4]:
AGE9_LABELS = {
    2: "16-24",
    3: "25-34",
    4: "35-44",
    5: "45-54",
    6: "55-64",
    7: "65-74",
    8: "75-84",
    9: "85+",
}

VOL_BINARY_A_LABELS = {
    0: "not_twiceplus_volunteer",
    1: "twiceplus_volunteer",
}

VOL_FREQ_5CAT_B_LABELS = {
    0: "not_volunteered",
    1: "once_or_one_off",
    2: "a_few_times",
    3: "monthly",
    4: "weekly",
}

ACTIVITY_LEVEL_LABELS = {
    0: "inactive",
    1: "fairly_active",
    2: "active",
}


def add_q5_variables(df):
    df = df.copy()

    # 年龄组
    df["age_group"] = df["Age9"].map(AGE9_LABELS)
    
    early_years = df["year"].isin([3, 4])
    later_years = df["year"].isin([5, 6, 7, 8])
    
    # Version A: 六年可比二分类志愿活跃度
    df["vol_binary_A"] = np.nan

    # Year 3-4:
    df.loc[
        early_years & (df["VolFrqB_Pop"] == 1),
        "vol_binary_A"
    ] = 1

    df.loc[
        early_years & (df["VolFrqB_Pop"] == 0),
        "vol_binary_A"
    ] = 0

    # Year 5-8: 2,3,4 表示几次/每月/每周，统一归为较活跃志愿者
    df.loc[
        later_years & (df["VolFrqB_Pop"].isin([2, 3, 4])),
        "vol_binary_A"
    ] = 1

    df.loc[
        later_years & (df["VolFrqB_Pop"].isin([0, 1])),
        "vol_binary_A"
    ] = 0

    df["vol_binary_A_label"] = df["vol_binary_A"].map(
        VOL_BINARY_A_LABELS
    )

    # Version B: year 5-8 原始五分类志愿频率
    df["vol_freq_5cat_B"] = np.nan

    df.loc[
        later_years & (df["VolFrqB_Pop"].isin([0, 1, 2, 3, 4])),
        "vol_freq_5cat_B"
    ] = df.loc[
        later_years & (df["VolFrqB_Pop"].isin([0, 1, 2, 3, 4])),
        "VolFrqB_Pop"
    ]

    df["vol_freq_5cat_B_label"] = df["vol_freq_5cat_B"].map(
        VOL_FREQ_5CAT_B_LABELS
    )
    
    # 活动参与活跃度：原始三分类
    df["activity_level_3cat"] = np.nan

    df.loc[
        df["MEMS7GR_ALL"].isin([0, 1, 2]),
        "activity_level_3cat"
    ] = df.loc[
        df["MEMS7GR_ALL"].isin([0, 1, 2]),
        "MEMS7GR_ALL"
    ]

    df["activity_level_3cat_label"] = df["activity_level_3cat"].map(
        ACTIVITY_LEVEL_LABELS
    )

    # 去掉没有有效年龄组的样本
    df = df[df["age_group"].notna()].copy()

    print(f"Done: {len(df)} rows after adding Q5 variables")

    print("vol_binary_A:")
    print(df["vol_binary_A"].value_counts(dropna=False).sort_index())

    print("vol_freq_5cat_B:")
    print(df["vol_freq_5cat_B"].value_counts(dropna=False).sort_index())

    print("activity_level_3cat:")
    print(df["activity_level_3cat"].value_counts(dropna=False).sort_index())

    return df

q5_master = add_q5_variables(q5_master)

q5_master.to_csv(
    OUTPUT_DIR / "q5_respondent_master_with_variables.csv",
    index=False
)

Done: 95750 rows after adding Q5 variables
vol_binary_A:
vol_binary_A
0.0    64443
1.0     7493
NaN    23814
Name: count, dtype: int64
vol_freq_5cat_B:
vol_freq_5cat_B
0.0    44145
1.0     1986
2.0     2143
3.0     1466
4.0     1879
NaN    44131
Name: count, dtype: int64
activity_level_3cat:
activity_level_3cat
0.0    21430
1.0    10267
2.0    64053
Name: count, dtype: int64


In [5]:
def weighted_mean(values, weights):
    mask = values.notna() & weights.notna()

    if mask.sum() == 0:
        return np.nan

    if weights[mask].sum() == 0:
        return np.nan

    return np.average(values[mask], weights=weights[mask])


def weighted_category_rate(data, value_col, category_value, weight_col):
    values = data[value_col]
    weights = data[weight_col]

    mask = values.notna() & weights.notna()

    if mask.sum() == 0:
        return np.nan

    if weights[mask].sum() == 0:
        return np.nan

    indicator = (
        values.loc[mask] == category_value
    ).astype(float)

    return np.average(
        indicator,
        weights=weights.loc[mask]
    )

构建数据集

In [6]:
def make_q5_borough_age_volunteer_panel(
    df,
    output_path,
    volunteer_col,
    volunteer_label_col,
    weight_col=WEIGHT_COL,
    min_cell_n=30
):
    rows = []

    group_cols = [
        "year",
        "LA_2023",
        "age_group",
    ]

    valid_df = df[
        df[volunteer_col].notna()
        & df[weight_col].notna()
    ].copy()

    for group_values, group_data in valid_df.groupby(group_cols, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        base_row = dict(zip(group_cols, group_values))

        total_weight = group_data[weight_col].sum()
        total_n = len(group_data)

        for volunteer_value, volunteer_data in group_data.groupby(
            volunteer_col,
            dropna=False
        ):
            row = base_row.copy()

            row["volunteer_category"] = volunteer_value

            if volunteer_label_col in volunteer_data.columns:
                row["volunteer_category_label"] = (
                    volunteer_data[volunteer_label_col]
                    .dropna()
                    .iloc[0]
                    if volunteer_data[volunteer_label_col].notna().any()
                    else np.nan
                )
            else:
                row["volunteer_category_label"] = np.nan

            if total_weight > 0:
                row["volunteer_category_rate"] = (
                    volunteer_data[weight_col].sum()
                    / total_weight
                )
            else:
                row["volunteer_category_rate"] = np.nan

            row["inactive_rate"] = weighted_category_rate(
                data=volunteer_data,
                value_col="activity_level_3cat",
                category_value=0,
                weight_col=weight_col
            )

            row["fairly_active_rate"] = weighted_category_rate(
                data=volunteer_data,
                value_col="activity_level_3cat",
                category_value=1,
                weight_col=weight_col
            )

            row["active_rate"] = weighted_category_rate(
                data=volunteer_data,
                value_col="activity_level_3cat",
                category_value=2,
                weight_col=weight_col
            )

            row["mean_number_activities_150"] = weighted_mean(
                volunteer_data["Number_Activities_150"],
                volunteer_data[weight_col]
            )

            row["n_parent_cell"] = total_n
            row["weighted_n_parent_cell"] = total_weight

            row["n"] = len(volunteer_data)
            row["weighted_n"] = volunteer_data[weight_col].sum()

            row["small_cell"] = row["n"] < min_cell_n

            rows.append(row)

    result = pd.DataFrame(rows)

    result = result.sort_values(
        [
            "year",
            "LA_2023",
            "age_group",
            "volunteer_category",
        ]
    ).reset_index(drop=True)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)

    print(f"Done: {len(result)} rows")
    print("Rows by year:")
    print(result["year"].value_counts().sort_index())

    print("Small cells:")
    print(result["small_cell"].value_counts(dropna=False))

    return result

version A dataset

In [7]:
q5_versionA = q5_master[
    q5_master["year"].isin([3, 4, 5, 6, 7, 8])
].copy()


q5_versionA_borough_age_panel = make_q5_borough_age_volunteer_panel(
    df=q5_versionA,
    output_path=OUTPUT_DIR / "q5_versionA_borough_age_binary_panel.csv",
    volunteer_col="vol_binary_A",
    volunteer_label_col="vol_binary_A_label",
    weight_col=WEIGHT_COL
)

Done: 2725 rows
Rows by year:
year
3    426
4    426
5    469
6    466
7    467
8    471
Name: count, dtype: int64
Small cells:
small_cell
True     1974
False     751
Name: count, dtype: int64


version B dataset

In [8]:
q5_versionB = q5_master[
    q5_master["year"].isin([5, 6, 7, 8])
].copy()


q5_versionB_borough_age_panel = make_q5_borough_age_volunteer_panel(
    df=q5_versionB,
    output_path=OUTPUT_DIR / "q5_versionB_borough_age_5cat_panel.csv",
    volunteer_col="vol_freq_5cat_B",
    volunteer_label_col="vol_freq_5cat_B_label",
    weight_col=WEIGHT_COL
)

Done: 3775 rows
Rows by year:
year
5     952
6     876
7     947
8    1000
Name: count, dtype: int64
Small cells:
small_cell
True     3141
False     634
Name: count, dtype: int64


check

In [9]:
print("Version A:")
print(q5_versionA_borough_age_panel.shape)
print(q5_versionA_borough_age_panel.head())

print("Version B:")
print(q5_versionB_borough_age_panel.shape)
print(q5_versionB_borough_age_panel.head())

print("Version A years:")
print(q5_versionA_borough_age_panel["year"].value_counts().sort_index())

print("Version B years:")
print(q5_versionB_borough_age_panel["year"].value_counts().sort_index())

print("Version A volunteer categories:")
print(
    q5_versionA_borough_age_panel[
        ["volunteer_category", "volunteer_category_label"]
    ]
    .drop_duplicates()
    .sort_values("volunteer_category")
)

print("Version B volunteer categories:")
print(
    q5_versionB_borough_age_panel[
        ["volunteer_category", "volunteer_category_label"]
    ]
    .drop_duplicates()
    .sort_values("volunteer_category")
)

print("Version A small cells:")
print(
    q5_versionA_borough_age_panel["small_cell"]
    .value_counts(dropna=False)
)

print("Version B small cells:")
print(
    q5_versionB_borough_age_panel["small_cell"]
    .value_counts(dropna=False)
)

Version A:
(2725, 15)
   year  LA_2023 age_group  volunteer_category volunteer_category_label  \
0     3      8.0     16-24                 0.0  not_twiceplus_volunteer   
1     3      8.0     16-24                 1.0      twiceplus_volunteer   
2     3      8.0     25-34                 0.0  not_twiceplus_volunteer   
3     3      8.0     25-34                 1.0      twiceplus_volunteer   
4     3      8.0     35-44                 0.0  not_twiceplus_volunteer   

   volunteer_category_rate  inactive_rate  fairly_active_rate  active_rate  \
0                 0.987418       0.380167            0.093848     0.525985   
1                 0.012582       0.000000            0.000000     1.000000   
2                 0.960536       0.453574            0.019692     0.526733   
3                 0.039464       0.000000            0.000000     1.000000   
4                 0.943492       0.387983            0.231152     0.380865   

   mean_number_activities_150  n_parent_cell  weighted_n_p